In [33]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split


In [16]:
import warnings
warnings.filterwarnings("ignore")

In [17]:
class CFG:
    TARGET = ['GGFM_f']
    N_FOLDS = 5
    RANDOM_STATE = 3

    COLORADO_PATH = './Data/Synt/Colorado_600_merge.csv'
    NSO_PATH = './Data/Synt/NSO_600_merge.csv'

In [18]:
class DataLoader:
    def __init__(self, colorado: pd.DataFrame, nso: pd.DataFrame):
        self.colorado = colorado
        self.nso = nso
        self.log_features = []  # Список признаков для трансформации
        self.X = None
        self.y = None
    

    @staticmethod
    def reduce_mem_usage(dataframe):
        """ 
        Уменьшает использование памяти dataframe путем преобразования типов данных
        с автоматическим пропуском временных столбцов
        """
        start_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Изначальное использование памяти: {start_mem:.2f} MB")
        
        for col in dataframe.columns:
            col_type = dataframe[col].dtype
            
            # Пропускаем временные столбцы и категориальные данные
            if str(col_type).startswith('datetime') or str(col_type) == 'category':
                continue
                
            if col_type != object:
                c_min = dataframe[col].min()
                c_max = dataframe[col].max()
                
                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        dataframe[col] = dataframe[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        dataframe[col] = dataframe[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        dataframe[col] = dataframe[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        dataframe[col] = dataframe[col].astype(np.int64)
                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        dataframe[col] = dataframe[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        dataframe[col] = dataframe[col].astype(np.float32)
                    else:
                        dataframe[col] = dataframe[col].astype(np.float64)
            else:
                # Оптимизация строковых столбцов
                dataframe[col] = dataframe[col].astype('category')
        
        end_mem = dataframe.memory_usage().sum() / 1024**2
        print(f"Итоговое использование памяти: {end_mem:.2f} MB")
        print(f"Экономия {(start_mem - end_mem) / start_mem * 100:.1f}%")
        
        return dataframe


    def load(self, option='colorado'):
        print(f'Loading data')
        print(f'Choosed option:{option}')

        if option == 'compare':
            self.colorado = self.colorado.merge(self.nso, left_on='latitude(deg)', right_on='lat(deg)', how='outer')
        elif option == 'colorado':
            pass

        self.colorado = self.reduce_mem_usage(self.colorado)

In [19]:
class DataAnalysis:

    @staticmethod
    def info_df(df: pd.DataFrame) -> None:
        print('------------------------------')
        print('| Dataset information |')
        print('------------------------------')
        df.info()
        print('-----------------------------------------')
        print('| First 5 rows |')
        print('-----------------------------------------')
        display(df.head())
        print('--------------------')
        print('| Sum of duplicates |')
        print('--------------------')
        print(df.duplicated().sum())


    @staticmethod
    def view_distribution(data: pd.DataFrame, object_col = False, numeric_col = False) -> None:
        numeric_cols = data.select_dtypes(exclude=['object', 'datetime']).columns.to_list()
        object_cols = data.select_dtypes(include=['object']).columns.to_list()
        
        if numeric_col:
            fig, axes = plt.subplots(nrows=len(data[numeric_cols].columns), ncols=2, figsize=(len(numeric_cols)+15,len(numeric_cols)+7))
            j = 0
            for i in data[numeric_cols].columns:
                sns.histplot(data[numeric_cols][i], ax=axes[j, 0], kde=True, bins=40, edgecolor='black')
                axes[j, 0].set_title(i, fontsize=14)
                axes[j, 0].set_xlabel('')

                sns.boxplot(x=data[numeric_cols][i], ax=axes[j, 1], orient='h', palette='pink')
                axes[j, 1].set_title(i, fontsize=14)
                axes[j, 1].set_xlabel('')
                j += 1
            plt.suptitle(f'Num features\n\n', ha='center', fontweight='bold', fontsize=20);
            plt.tight_layout();
            plt.show();

        if object_col:
            _,ax = plt.subplots(len(object_cols),1, figsize=(len(object_cols)+7,len(object_cols)+20));
            ax =ax.flatten();
            g = 0
            for k in data[object_cols].columns:
                sns.countplot(data=data, x=k,ax=ax[g]);
                ax[g].set_xticklabels(labels=ax[g].get_xticklabels());
                ax[g].set_title(k);
                ax[g].set_xlabel('');
                g += 1
            plt.suptitle(f'Categorical\n\n', ha='center', fontweight='bold', fontsize=20);
            plt.show();


    @staticmethod
    def bloating_of_variance(data: pd.DataFrame) -> None:
        num = data.select_dtypes(exclude=['object', 'datetime']).columns.to_list()
        vif_data = pd.DataFrame()
        vif_data['feature'] = data.select_dtypes(exclude=['object', 'datetime']).columns.to_list()

        vif_data['VIF'] = [variance_inflation_factor(data[num].values, i) \
                                for i in range(len(data[num].columns))]
        print(vif_data)
    
    @staticmethod
    def balance_of_target(data: pd.DataFrame, target: str) -> None:
        sns.countplot(y=target, data=data, color='green', width=0.6);

    @staticmethod
    def plot_count(df: pd.core.frame.DataFrame, col: str, title_name: str='Train') -> None:
        # Set background color
        f, ax = plt.subplots(1, 2, figsize=(16, 7))
        plt.subplots_adjust(wspace=0.2)

        s1 = df[col].value_counts()
        N = len(s1)

        outer_sizes = s1
        inner_sizes = s1/N

        colors = sns.color_palette("mako")
        # hex_colors = [matplotlib.colors.to_hex(color) for color in colors]
        # print(hex_colors)
        
        outer_colors = ['#2e1e3b', '#413d7b', '#37659e', '#348fa7', '#40b7ad', '#8bdab2']
        inner_colors = ['#2e1e3b', '#413d7b', '#37659e', '#348fa7', '#40b7ad', '#8bdab2']
        #inner_colors = ['#59b3a3',] #'#433C64']

        ax[0].pie(
            outer_sizes,colors=outer_colors, 
            labels=s1.index.tolist(), 
            startangle=90, frame=True, radius=1.3, 
            explode=([0.05]*(N-1) + [.3]),
            wedgeprops={'linewidth' : 1, 'edgecolor' : 'black'}, 
            textprops={'fontsize': 12, 'weight': 'bold', 'color': 'white'}
        )

        textprops = {
            'size': 13, 
            'weight': 'bold', 
            'color': 'white'
        }

        ax[0].pie(
            inner_sizes, colors=inner_colors,
            radius=1, startangle=90,
            autopct='%1.f%%', explode=([.1]*(N-1) + [.3]),
            pctdistance=0.8, textprops=textprops
        )

        center_circle = plt.Circle((0,0), .68, color='black', fc='#243139', linewidth=0)
        ax[0].add_artist(center_circle)

        x = s1
        y = s1.index.tolist()
        sns.barplot(
            x=x, y=y, ax=ax[1],
            palette=colors, orient='horizontal'
        )

        ax[1].spines['top'].set_visible(False)
        ax[1].spines['right'].set_visible(False)
        ax[1].tick_params(
            axis='x',         
            which='both',      
            bottom=False,       
            labelbottom=False
        )

        for i, v in enumerate(s1):
            ax[1].text(v, i+0.1, str(v), color='white', fontweight='bold', fontsize=12)

        plt.setp(ax[1].get_yticklabels(), fontweight="bold")
        plt.setp(ax[1].get_xticklabels(), fontweight="bold")
        ax[1].set_xlabel(col, fontweight="bold", color='white')
        ax[1].set_ylabel('count', fontweight="bold", color='white')

        f.suptitle(f'{title_name}', fontsize=14, fontweight='bold', color='white')
        plt.tight_layout() 
        plt.show()
    
    @staticmethod
    def summary(data: pd.DataFrame) -> None:
        data = data.select_dtypes(exclude=['object', 'datetime'])
        sum = pd.DataFrame(data.dtypes, columns=['dtypes'])
        sum['missing#'] = data.isna().sum()
        sum['missing%'] = (data.isna().sum())/len(data)
        sum['uniques'] = data.nunique().values
        sum['count'] = data.count().values
        sum['skew'] = data.skew().values
        return sum
    
    @staticmethod
    def correlations(data: pd.DataFrame) -> None:
        data = data.drop(columns=CFG.TARGET)
        plt.figure(figsize=(15, 13));
        # Generate a mask for the upper triangle
        mask_pir = np.triu(np.ones_like(data.corr(method='pearson'), dtype=bool));
        mask_spi = np.triu(np.ones_like(data.corr(method='spearman'), dtype=bool));
       
        # Set up the matplotlib figure
        f, ax = plt.subplots(figsize=(11, 9));

        # Generate a custom diverging colormap
        cmap = sns.diverging_palette(230, 20, as_cmap=True);
        plt.title('PIRSON')
        sns.heatmap(data.corr(method='pearson'), annot=True, mask=mask_pir, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, robust=True);
        plt.show();
        
        plt.figure(figsize=(15, 13));
        f, ax = plt.subplots(figsize=(11, 9));
        plt.title('SPEARMAN')
        sns.heatmap(data.corr(method='spearman'), annot=True, mask=mask_spi, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, robust=True)
        plt.show();

        plt.figure(figsize=(15, 13));
        f, ax = plt.subplots(figsize=(11, 9));
        
        interval_cols = data.select_dtypes(exclude='object').columns.to_list()
        phik_overview = data.phik_matrix(interval_cols=interval_cols)
        plt.title(r'$\phi_K$')
        corr = phik_overview.round(2)
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, cmap='pink_r', vmax=.3, center=0,
                annot=True, fmt='.2f', square=True, linewidths=.5, cbar_kws={"shrink": .5})

        significance_overview  = data.significance_matrix(interval_cols=interval_cols)

        plt.figure(figsize=(15, 13));
        plt.title('Statistical significance')
        corr = significance_overview.round(2)
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, cmap='pink_r', vmax=5, vmin=-5, center=0,
                annot=True, fmt='.2f', square=True, linewidths=.5, cbar_kws={"shrink": .5})

        plt.show()
    
    @staticmethod
    def blinks(data: pd.DataFrame) -> None:
        print('Data gaps')
        missingno.matrix(data)

In [20]:
loader = DataLoader(
    colorado=pd.read_csv(CFG.COLORADO_PATH,
                        skipinitialspace=True,
                        index_col='index'),
    nso=pd.read_csv(CFG.NSO_PATH,
                        skipinitialspace=True,
                        index_col='index'),
)

In [21]:
loader.load()

Loading data
Choosed option:colorado
Изначальное использование памяти: 0.46 MB
Итоговое использование памяти: 0.17 MB
Экономия 63.5%


In [22]:
loader.colorado.head()

,Latitude (deg),Longitude (deg),Height (m),Height relief (m),Height geoid (m),r (m),GGFM_f,GGFM_g,TGFM_f,TGFM_g,Density (kg/m^3),Density STD (kg/m^3)
index,,,,,,,,,,,,
0,35.0,250.0,1614.0,1638.0,-24.46875,6386785.5,62410340.0,9.773438,62393676.0,9.773438,2412.0,270.5
1,35.0,250.0,1814.0,1638.0,-24.46875,6386985.5,62408384.0,9.773438,62391724.0,9.773438,2412.0,270.5
2,35.0,250.0,2114.0,1638.0,-24.46875,6387285.5,62405452.0,9.773438,62388792.0,9.773438,2412.0,270.5
3,35.0,250.0,2614.0,1638.0,-24.46875,6387785.5,62400568.0,9.765625,62383908.0,9.765625,2412.0,270.5
4,35.0,250.0,4200.0,1638.0,-24.46875,6389372.0,62385076.0,9.765625,62368412.0,9.765625,2412.0,270.5


In [23]:
loader.colorado.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4608 entries, 0 to 4607
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Latitude (deg)        4608 non-null   float16
 1   Longitude (deg)       4608 non-null   float16
 2   Height (m)            4608 non-null   float16
 3   Height relief (m)     4608 non-null   float16
 4   Height geoid (m)      4608 non-null   float16
 5   r (m)                 4608 non-null   float32
 6   GGFM_f                4608 non-null   float32
 7   GGFM_g                4608 non-null   float16
 8   TGFM_f                4608 non-null   float32
 9   TGFM_g                4608 non-null   float16
 10  Density (kg/m^3)      4608 non-null   float16
 11  Density STD (kg/m^3)  4608 non-null   float16
dtypes: float16(9), float32(3)
memory usage: 171.0 KB


In [24]:
# for data in [loader.colorado]:
#     DataAnalysis.info_df(data)
#     DataAnalysis.blinks(data)
#     DataAnalysis.view_distribution(data, numeric_col=True)
#     DataAnalysis.correlations(data)

In [25]:
DataAnalysis.summary(loader.colorado).style.background_gradient(cmap='Blues')

,dtypes,missing#,missing%,uniques,count,skew
Latitude (deg),float16,0,0.000000,24,4608,nan
Longitude (deg),float16,0,0.000000,24,4608,inf
Height (m),float16,0,0.000000,1279,4608,-inf
Height relief (m),float16,0,0.000000,492,4608,inf
Height geoid (m),float16,0,0.000000,415,4608,inf
r (m),float32,0,0.000000,2032,4608,-0.237388
GGFM_f,float32,0,0.000000,3476,4608,0.200883
GGFM_g,float16,0,0.000000,4,4608,inf
TGFM_f,float32,0,0.000000,3873,4608,0.202297
TGFM_g,float16,0,0.000000,4,4608,inf


In [26]:
loader.colorado.describe().T\
            .style.bar(subset=['mean'], color=px.colors.qualitative.G10[2])\
            .background_gradient(subset=['std'], cmap='Blues')\
            .background_gradient(subset=['50%'], cmap='BuGn')

,count,mean,std,min,25%,50%,75%,max
Latitude (deg),4608.000000,inf,1.500977,35.000000,36.257812,37.500000,38.742188,40.000000
Longitude (deg),4608.000000,inf,2.408203,250.000000,252.031250,254.000000,255.968750,258.000000
Height (m),4608.000000,inf,inf,956.000000,2310.000000,4200.000000,4700.000000,5200.000000
Height relief (m),4608.000000,inf,inf,982.500000,1494.750000,1903.500000,2334.500000,3874.000000
Height geoid (m),4608.000000,-inf,3.173828,-26.906250,-21.535156,-19.898438,-17.089844,-12.562500
r (m),4608.000000,6389574.000000,1388.661499,6386270.500000,6388385.500000,6389802.000000,6390724.000000,6392176.000000
GGFM_f,4608.000000,62378928.000000,14807.063477,62349224.000000,62367420.000000,62377624.000000,62390465.000000,62415344.000000
GGFM_g,4608.000000,9.765625,0.006718,9.750000,9.757812,9.757812,9.765625,9.773438
TGFM_f,4608.000000,62362896.000000,14706.228516,62333312.000000,62351402.000000,62361654.000000,62374479.000000,62399044.000000
TGFM_g,4608.000000,9.765625,0.006191,9.750000,9.757812,9.765625,9.765625,9.773438


In [27]:
loader.colorado.drop(columns=['GGFM_g'], inplace=True)
loader.colorado.drop_duplicates(inplace=True)
loader.colorado.reset_index(drop=True, inplace=True)

In [28]:
features = loader.colorado.drop(columns=CFG.TARGET)
target = loader.colorado[CFG.TARGET]

# CatBoost

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, random_state=CFG.RANDOM_STATE
)

In [30]:
class LaplaceLossObjective:
    def calc_ders_range(self, approxes, targets, weights):
        """
        Кастомная функция потерь, учитывающая условия Лапласа
        Комбинация MSE и регуляризации на лапласиан
        """
        assert len(approxes) == len(targets)
        if weights is not None:
            assert len(weights) == len(approxes)
        
        result = []
        for i in range(len(approxes)):
            # Основная ошибка (MSE)
            error = approxes[i] - targets[i]
            
            # Первая производная (градиент)
            grad = 2.0 * error
            
            # Вторая производная (гессиан)
            hess = 2.0
            
            # Добавляем регуляризацию на лапласиан (условия Лапласа)
            # Это упрощенная реализация - в реальности нужно вычислять лапласиан
            # от приближения в пространстве координат
            laplace_regularization = 0.01  # Коэффициент регуляризации
            
            # Модифицируем градиент и гессиан
            grad += laplace_regularization * approxes[i]  # Упрощенная регуляризация
            hess += laplace_regularization
            
            if weights is not None:
                grad *= weights[i]
                hess *= weights[i]
            
            result.append((grad, hess))
        
        return result

In [31]:
def calculate_laplacian(predictions, coordinates):
    """
    Вычисляет дискретный лапласиан для набора предсказаний
    coordinates должен содержать пространственные координаты
    """
    # Это упрощенная реализация - в реальности нужно использовать
    # конечные разности или другие методы для вычисления лапласиана
    laplacian = np.zeros_like(predictions)
    
    # Здесь должна быть реализация вычисления лапласиана
    # на основе пространственных координат
    
    return laplacian

In [34]:
def objective(trial):
    # Параметры для оптимизации
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations', 1, 10),
        'verbose': False,
        'task_type': 'GPU',
    }
    
    # Инициализация модели с кастомной функцией потерь
    model = CatBoostRegressor(
        loss_function=LaplaceLossObjective(),
        **params
    )
    
    # Обучение модели
    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        early_stopping_rounds=50,
        verbose=False
    )
    
    # Предсказание и оценка качества
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    
    return mse

In [37]:
def train_laplace_catboost(X, y):
    # Исследование гиперпараметров с Optuna
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50)
    
    print("Лучшие гиперпараметры:")
    print(study.best_params)
    print(f"Лучшее MSE: {study.best_value}")
    
    # Обучение финальной модели с лучшими параметрами
    best_params = study.best_params
    best_params['verbose'] = False
    
    final_model = CatBoostRegressor(
        loss_function=LaplaceLossObjective(),
        **best_params
    )
    
    # Обучение на всех данных
    final_model.fit(X, y, verbose=False)
    
    # Сохранение модели
    final_model.save_model('laplace_catboost_model.cbm')
    
    return final_model, study.best_params


In [38]:
def predict_at_point(model, latitude, longitude, height, height_relief, 
                    height_geoid, r, density, density_std):
    """
    Предсказание потенциала в произвольной точке
    """
    point_data = np.array([[latitude, longitude, height, height_relief, 
                           height_geoid, r, density, density_std]])
    
    prediction = model.predict(point_data)
    return prediction[0]

In [39]:
def evaluate_laplace_condition(model, X_data):
    """
    Оценка выполнения условия Лапласа для предсказаний модели
    """
    predictions = model.predict(X_data)
    
    # Вычисляем лапласиан (упрощенная версия)
    # В реальности нужно использовать пространственные координаты
    coordinates = X_data[['Latitude (deg)', 'Longitude (deg)', 'Height (m)']].values
    
    # Здесь должна быть реализация вычисления лапласиана
    # Для демонстрации используем упрощенный подход
    laplacian_approx = np.zeros_like(predictions)
    
    # Оцениваем, насколько хорошо выполняется условие Лапласа (Δφ = 0)
    laplace_violation = np.mean(np.abs(laplacian_approx))
    
    return laplace_violation


In [41]:
model, best_params = train_laplace_catboost(X_train,y_train)

[I 2025-08-28 19:12:21,124] A new study created in memory with name: no-name-939223ba-48f3-4af1-a070-901e37c8194f
Got unsafe target value = 6.24134e+07 at object #0 of dataset learn
Got unsafe target value = 6.23622e+07 at object #0 of dataset test #0
[W 2025-08-28 19:12:21,821] Trial 0 failed with parameters: {'iterations': 909, 'depth': 10, 'learning_rate': 0.2185987530151272, 'l2_leaf_reg': 5.925157456405029, 'border_count': 57, 'random_strength': 7.473691735009707, 'bagging_temperature': 0.06650192848825465, 'leaf_estimation_iterations': 3} because of the following error: CatBoostError('catboost/libs/metrics/metric.cpp:6723: If loss function is a user defined object, then the eval metric must be specified.').
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\DS\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\SerKer\AppData\Local\Temp\ipykernel_17236\36600976.py", line 23, in objective
  

CatBoostError: catboost/libs/metrics/metric.cpp:6723: If loss function is a user defined object, then the eval metric must be specified.